In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import r2_score

In [2]:
datasetPath = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/notrees-dl.csv")

#making sure
datasetPath.is_file()

True

In [3]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.187436,0.000161,0.000196,1124.69017,1020.232214,0.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.116122
1,0.092571,0.000277,0.000303,1124.69017,1020.232214,1.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.106717
2,0.057824,0.000337,0.000359,1124.69017,1020.232214,2.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.097078
3,0.171885,0.000224,0.000273,1124.69017,1020.232214,3.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.112769
4,0.110673,0.000253,0.000287,1124.69017,1020.232214,4.0,180.0,-71.945084,19.489212,0.0,-19.258,-0.0,0.103993


In [4]:
target_col = dataset["grviOut"]
labels = target_col.to_numpy(dtype="float32")
labels

array([0.1161218 , 0.10671729, 0.09707835, ..., 0.13551866, 0.13556816,
       0.14273934], shape=(329620,), dtype=float32)

In [5]:
inputs = dataset.drop("grviOut", axis=1)
inputs = inputs.to_numpy(dtype="float32")
inputs

array([[ 1.8743615e-01,  1.6062829e-04,  1.9582233e-04, ...,
         0.0000000e+00, -1.9257999e+01, -0.0000000e+00],
       [ 9.2570759e-02,  2.7738232e-04,  3.0346971e-04, ...,
         0.0000000e+00, -1.9257999e+01, -0.0000000e+00],
       [ 5.7823673e-02,  3.3685658e-04,  3.5936269e-04, ...,
         0.0000000e+00, -1.9257999e+01, -0.0000000e+00],
       ...,
       [ 2.3189114e-01,  1.0695464e-03,  1.6404410e-03, ...,
        -0.0000000e+00, -2.2218000e+01, -1.8000000e+02],
       [ 2.3585278e-01,  8.9433161e-04,  1.3592206e-03, ...,
        -0.0000000e+00, -2.2218000e+01, -1.8000000e+02],
       [ 2.4523808e-01,  8.7076728e-04,  1.3455153e-03, ...,
        -0.0000000e+00, -2.2218000e+01, -1.8000000e+02]],
      shape=(329620, 12), dtype=float32)

In [6]:
if len(labels) != len (inputs):
    print ("problem")

In [7]:
n_total = len(inputs)

n_train = int(0.8*n_total)
n_test = int(0.1*n_total)
n_val = int(0.1*n_total)

# shuffle
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_total, generator=g).numpy()

inputs_shuf = inputs[perm]
labels_shuf = labels[perm]

# split
train_inputs, train_labels = inputs_shuf[:n_train], labels_shuf[:n_train]
val_inputs, val_labels = inputs_shuf[n_train:n_train + n_val], labels_shuf[n_train:n_train + n_val]
test_inputs, test_labels = inputs_shuf[n_train + n_val:], labels_shuf[n_train + n_val:]

In [8]:
train_inputs = torch.from_numpy(train_inputs).float()
train_labels = torch.from_numpy(train_labels).float()

val_inputs = torch.from_numpy(val_inputs).float()
val_labels = torch.from_numpy(val_labels).float()

test_inputs = torch.from_numpy(test_inputs).float()
test_labels = torch.from_numpy(test_labels).float()

In [9]:
print ("Train size:", len(train_inputs))
print ("Val size:", len(val_inputs))
print ("Test size:", len(test_inputs))

Train size: 263696
Val size: 32962
Test size: 32962


In [10]:
BATCH_SIZE = 512

class metricsDataset (Dataset):
    def __init__(self, metrics, labels):
        self.metrics = metrics
        self.labels = labels

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], self.labels[idx]

train_dataset = metricsDataset(train_inputs, train_labels)
val_dataset   = metricsDataset(val_inputs, val_labels)
test_dataset  = metricsDataset(test_inputs, test_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [12]:
model = nn.Sequential(
    nn.Linear(12, 128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.ReLU(),

    nn.Linear(64, 1)
).to(device)

In [13]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

In [14]:
EPOCHS = 150

best_val_loss = float("inf")

for epoch in range(0, EPOCHS):
    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        optimizer.zero_grad()

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        preds = model(x).squeeze(-1)
        loss = criterion(preds, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            pred = model(x).squeeze(-1)
            loss = criterion(pred, y)

            val_loss += loss.item() * x.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoch {epoch + 1:3d}/{EPOCHS} | "
        f"Train MSE: {train_loss:.6f} | "
        f"Val MSE: {val_loss:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        # SAVE the model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, 'best_model_checkpoint.pth')

        print("Best Model Saved")

checkpoint = torch.load('best_model_checkpoint.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model (Val Loss: {checkpoint['best_val_loss']})")

model.eval()
all_preds = []
all_targets = []
test_loss = 0.0
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x).squeeze(-1)
        loss = criterion(pred, y)
        test_loss += loss.item() * x.size(0)
        all_preds.append(pred.cpu().numpy())
        all_targets.append(y.cpu().numpy())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

test_loss /= len(test_loader.dataset)
r2 = r2_score(all_targets, all_preds)

rel_test_loss = ((test_loss ** 0.5)*100) / (all_targets.max()-all_targets.min())

print(f"\nTest MSE: {test_loss:.6f}")
print(f"Test RMSE: {test_loss ** 0.5:.6f}")
print(str(float(rel_test_loss)) + "%")
print(f"R2: {r2:.6f}")

Epoch   1/150 | Train MSE: 1.028667 | Val MSE: 0.009887
Best Model Saved
Epoch   2/150 | Train MSE: 0.006034 | Val MSE: 0.006088
Best Model Saved
Epoch   3/150 | Train MSE: 0.004145 | Val MSE: 0.002385
Best Model Saved
Epoch   4/150 | Train MSE: 0.097974 | Val MSE: 0.008433
Epoch   5/150 | Train MSE: 0.043586 | Val MSE: 0.150388
Epoch   6/150 | Train MSE: 0.068453 | Val MSE: 0.023402
Epoch   7/150 | Train MSE: 0.120947 | Val MSE: 0.002552
Epoch   8/150 | Train MSE: 0.002096 | Val MSE: 0.003935
Epoch   9/150 | Train MSE: 0.028678 | Val MSE: 0.001315
Best Model Saved
Epoch  10/150 | Train MSE: 0.049364 | Val MSE: 0.002318
Epoch  11/150 | Train MSE: 0.012079 | Val MSE: 0.001099
Best Model Saved
Epoch  12/150 | Train MSE: 0.025609 | Val MSE: 0.004837
Epoch  13/150 | Train MSE: 0.022713 | Val MSE: 0.008340
Epoch  14/150 | Train MSE: 0.017303 | Val MSE: 0.001102
Epoch  15/150 | Train MSE: 0.011552 | Val MSE: 0.002724
Epoch  16/150 | Train MSE: 0.016695 | Val MSE: 0.015544
Epoch  17/150 | Tra